# 07 · Embeddings de un modelo de lenguaje y más variantes de modelos

El modelo del `06_svd_modelos_stacking.ipynb` (promedio de **LightGBM** y **LightGBM con SVD del texto**) llega a **0,9457 en la validación** y **0,94614 en Kaggle**.

En el 06 se vio que lo que suma es un modelo que **mire la información de otra manera**. Este notebook prueba dos ideas en esa línea:

1. **Embeddings de un modelo de lenguaje preentrenado:** una forma de ver el texto por su **significado**, no por las palabras exactas.
2. **Variantes de LightGBM y CatBoost** que aprenden distinto: `extra_trees`, DART y CatBoost con la SVD.

Mismo criterio que siempre: comparación partición por partición contra el modelo 06; se acepta si mejora más que el ruido (~0,001) y en casi todas las particiones.

**Resultado:** ninguna de las dos ideas mejora más que el ruido. El modelo final sigue siendo el del 06.

Requiere `torch` y `sentence-transformers`. La primera vez baja el modelo de lenguaje (~470 MB). Tarda en total unas **2 horas**, la mitad en calcular los embeddings.

## 0. Datos, features, piezas y el modelo 06

Lo mismo que en el 06. El modelo 06 se recalcula como referencia. Tarda unos 20 minutos.

In [1]:
import time
import warnings
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

from features import cargar_datos, armar_tabla, calcular_piezas, armar_conjunto, texto_combinado, componentes_svd, CAT
warnings.filterwarnings("ignore", category=pd.errors.PerformanceWarning)

todo, y = cargar_datos()
N_TRAIN = len(y)
X_todo = armar_tabla(todo)
X_svd = pd.concat([X_todo, componentes_svd(todo, 128)], axis=1)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
PARTICIONES = list(cv.split(np.zeros(N_TRAIN), y)) + [(np.arange(N_TRAIN), np.arange(N_TRAIN, len(todo)))]

inicio = time.time()
piezas = [calcular_piezas(todo, y, idx_a, idx_b, caracteres=True, cruzados=True) for idx_a, idx_b in PARTICIONES[:5]]
print(X_svd.shape, f"piezas listas en {(time.time() - inicio) / 60:.1f} minutos")

(100000, 220) piezas listas en 6.6 minutos


In [2]:
PARAMS_LGBM = dict(n_estimators=1500, learning_rate=0.02, num_leaves=63, colsample_bytree=0.5, random_state=42, verbose=-1)

def oof(X, modelo=LGBMClassifier, params=PARAMS_LGBM, extra=None, preparar=None, **fit_kwargs):
    """Probabilidad out of fold. extra[f]: piezas adicionales para la partición f."""
    p = np.zeros(N_TRAIN)
    for f, (idx_a, idx_b) in enumerate(PARTICIONES[:5]):
        A, B = armar_conjunto(X, {**piezas[f], **(extra[f] if extra else {})}, idx_a, idx_b)
        if preparar is not None:
            A, B = preparar(A), preparar(B)
        p[idx_b] = modelo(**params).fit(A, y[idx_a], **fit_kwargs).predict_proba(B)[:, 1]
    return p

def accuracy_por_particion(p):
    return np.array([accuracy_score(y[idx_b], p[idx_b] > 0.5) for _, idx_b in PARTICIONES[:5]])

def desacuerdo(p, q):
    return ((p > 0.5) != (q > 0.5)).mean()

resultados = []
def comparar(nombre, p, ref):
    notas, notas_ref = accuracy_por_particion(p), accuracy_por_particion(ref)
    dif = notas - notas_ref
    resultados.append({"experimento": nombre, "accuracy": round(notas.mean(), 4), "vs. referencia": round(dif.mean(), 4),
                       "mejora en": f"{(dif > 0).sum()}/5", "diferencia por partición": " ".join(f"{d:+.4f}" for d in dif)})
    print(f"{nombre:60s} {notas.mean():.4f}  ({dif.mean():+.4f}, mejora en {(dif > 0).sum()}/5)")

p_lgbm, p_lgbm_svd = oof(X_todo), oof(X_svd)
p_06 = (p_lgbm + p_lgbm_svd) / 2
print(f"LightGBM {accuracy_por_particion(p_lgbm).mean():.4f} | LightGBM con SVD {accuracy_por_particion(p_lgbm_svd).mean():.4f} | "
      f"modelo 06 {accuracy_por_particion(p_06).mean():.4f}")

LightGBM 0.9444 | LightGBM con SVD 0.9452 | modelo 06 0.9457


## 1. Embeddings de un modelo de lenguaje

Un **embedding** es un vector de números que representa el **significado** de un texto. Lo calcula un modelo de lenguaje (una red neuronal) que fue entrenado previamente con millones de textos, de forma que textos con significado parecido quedan cerca: "sin uso" y "a estrenar", o "celular" y "smartphone", aunque no compartan palabras.

Se usa `paraphrase-multilingual-MiniLM-L12-v2` (de `sentence-transformers`): un modelo chico, multilingüe (entiende castellano), que convierte cada texto en **384 números**. Se aplica al texto combinado (título + garantía + atributos), cortado en 96 *tokens* (pedazos de palabra).

El modelo **no se entrena** con nuestros datos ni usa la respuesta: solo se aplica. Por eso se calcula una sola vez para train + test, como el TF-IDF. Tarda unos 30 minutos.

In [3]:
from sentence_transformers import SentenceTransformer

inicio = time.time()
lenguaje = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2", device="cpu")
lenguaje.max_seq_length = 96
E = lenguaje.encode(texto_combinado(todo).tolist(), batch_size=128, normalize_embeddings=True).astype(np.float32)
print(E.shape, f"listo en {(time.time() - inicio) / 60:.1f} minutos")

C:\Users\Ezequiel\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:  21%|██        | 42/199 [00:00<00:00, 398.65it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1404.23it/s]

(100000, 384) listo en 25.8 minutos


Tres formas de usarlos, todas probadas sobre el modelo 06:

- **`prob_emb`:** regresión logística sobre los 384 números → probabilidad de usado. Fuera de fold, igual que la del TF-IDF.
- **Vecinos por significado:** para cada publicación, las 10 más parecidas (similitud coseno entre embeddings) de **otros vendedores**, y qué proporción de ellas era usada (`emb_knn_media`), más qué tan parecida es la más cercana (`emb_knn_sim`). Fuera de fold.
- **Tercera vista:** un LightGBM con el embedding resumido en 64 columnas (PCA), para promediar con los dos del 06.

Tarda unos 15 minutos.

In [4]:
vendedor = todo["seller_id"].values
sub_cv = lambda: StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

def prob_emb(idx_a, idx_b):
    pa = np.zeros(len(idx_a))
    for a2, b2 in sub_cv().split(idx_a, y[idx_a]):
        pa[b2] = LogisticRegression(C=2, max_iter=3000).fit(E[idx_a[a2]], y[idx_a[a2]]).predict_proba(E[idx_a[b2]])[:, 1]
    return pa, LogisticRegression(C=2, max_iter=3000).fit(E[idx_a], y[idx_a]).predict_proba(E[idx_b])[:, 1]

def vecinos(idx_ref, idx_obj, K=10):
    salida, E_ref, y_ref, v_ref = np.zeros((len(idx_obj), 2), np.float32), E[idx_ref].T, y[idx_ref], vendedor[idx_ref]
    for i in range(0, len(idx_obj), 2000):
        sl = slice(i, i + 2000)
        sim = E[idx_obj[sl]] @ E_ref                                     # similitud coseno (los vectores tienen norma 1)
        sim[vendedor[idx_obj[sl]][:, None] == v_ref[None, :]] = -1       # fuera el mismo vendedor
        top = np.argpartition(-sim, K, axis=1)[:, :K]
        s = np.take_along_axis(sim, top, 1)
        w = np.clip(s, 0, None)
        salida[sl, 0] = (w * y_ref[top]).sum(1) / np.maximum(w.sum(1), 1e-6)
        salida[sl, 1] = s.max(1)
    return salida

inicio = time.time()
extra_emb = []
for idx_a, idx_b in PARTICIONES[:5]:
    fa = np.zeros((len(idx_a), 2), np.float32)
    for a2, b2 in sub_cv().split(idx_a, y[idx_a]):
        fa[b2] = vecinos(idx_a[a2], idx_a[b2])
    fb = vecinos(idx_a, idx_b)
    extra_emb.append({"prob_emb": prob_emb(idx_a, idx_b),
                      "emb_knn_media": (fa[:, 0], fb[:, 0]), "emb_knn_sim": (fa[:, 1], fb[:, 1])})
p_emb_sola = np.zeros(N_TRAIN)
for f, (_, idx_b) in enumerate(PARTICIONES[:5]):
    p_emb_sola[idx_b] = extra_emb[f]["prob_emb"][1]
print(f"listo en {(time.time() - inicio) / 60:.1f} minutos | regresión logística sobre el embedding, sola: "
      f"{accuracy_por_particion(p_emb_sola).mean():.4f}")

p_lgbm_e     = oof(X_todo, extra=extra_emb)
p_lgbm_svd_e = oof(X_svd, extra=extra_emb)
X_emb = pd.concat([X_todo, pd.DataFrame(PCA(n_components=64, random_state=42).fit_transform(E).astype(np.float32),
                                         columns=[f"emb_{j}" for j in range(64)])], axis=1)
p_vista_emb = oof(X_emb)

comparar("1. modelo 06 con prob_emb y vecinos en los dos LightGBM", (p_lgbm_e + p_lgbm_svd_e) / 2, p_06)
comparar("1. modelo 06 + tercer LightGBM con el embedding (PCA 64)", (p_lgbm + p_lgbm_svd + p_vista_emb) / 3, p_06)
print(f"el LightGBM con el embedding no coincide con LightGBM en el {desacuerdo(p_vista_emb, p_lgbm):.1%}")

listo en 6.6 minutos | regresión logística sobre el embedding, sola: 0.7427


1. modelo 06 con prob_emb y vecinos en los dos LightGBM      0.9453  (-0.0004, mejora en 1/5)
1. modelo 06 + tercer LightGBM con el embedding (PCA 64)     0.9454  (-0.0003, mejora en 1/5)
el LightGBM con el embedding no coincide con LightGBM en el 1.3%


**Lectura:** los embeddings **no suman**. La regresión logística sobre el embedding, sola, queda muy por debajo de la del TF-IDF (~0,84).

El motivo: el modelo de lenguaje fue entrenado para captar el **tema general** de una frase ("esto es un celular", "esto es ropa de bebé"). Para saber si algo es nuevo o usado importan **detalles puntuales** ("usado", "impecable", "con caja", "sellado", el modelo exacto), que el TF-IDF toma tal cual y el embedding diluye al resumir todo en 384 números. El modelo "más moderno" no es el mejor para este problema.

## 2. Variantes de LightGBM y CatBoost

Modelos que aprenden de forma distinta al LightGBM estándar, para sumarlos como **tercer modelo** al promedio del 06:

- **LightGBM con `extra_trees=True`:** en cada corte, en vez de buscar el mejor umbral, prueba uno al azar por columna. Los árboles quedan más distintos (y más débiles, por eso se usan el doble de árboles).
- **LightGBM DART:** durante el entrenamiento "apaga" al azar algunos de los árboles ya construidos, para que los nuevos no dependan tanto de los primeros.
- **CatBoost con la SVD:** CatBoost (árboles simétricos) con las mismas columnas que el LightGBM con SVD.

Tarda unos 40 minutos (CatBoost, 25).

In [5]:
def categoricas_como_texto(A):
    A = A.copy()
    for c in CAT:
        A[c] = A[c].astype(object).fillna("NA").astype(str)
    return A

variantes = {
    "LightGBM extra_trees":          oof(X_todo, params={**PARAMS_LGBM, "extra_trees": True, "n_estimators": 3000}),
    "LightGBM extra_trees con SVD":  oof(X_svd, params={**PARAMS_LGBM, "extra_trees": True, "n_estimators": 3000}),
    "LightGBM DART con SVD":         oof(X_svd, params={**PARAMS_LGBM, "boosting_type": "dart", "n_estimators": 800,
                                                         "learning_rate": 0.08, "drop_rate": 0.1}),
    "CatBoost con SVD":              oof(X_svd, CatBoostClassifier,
                                         dict(iterations=2000, learning_rate=0.06, depth=7, random_seed=42, verbose=0,
                                              thread_count=8, allow_writing_files=False),
                                         preparar=categoricas_como_texto, cat_features=CAT),
}
for nombre, p in variantes.items():
    comparar(f"2. {nombre}, solo (vs. modelo 06)", p, p_06)
    print(f"   no coincide con LightGBM en el {desacuerdo(p, p_lgbm):.1%}")
    comparar(f"2. modelo 06 + {nombre}", (p_lgbm + p_lgbm_svd + p) / 3, p_06)

comparar("2. modelo 06 + CatBoost con SVD + DART", (p_lgbm + p_lgbm_svd + variantes["CatBoost con SVD"]
                                                    + variantes["LightGBM DART con SVD"]) / 4, p_06)

2. LightGBM extra_trees, solo (vs. modelo 06)                0.9436  (-0.0021, mejora en 0/5)
   no coincide con LightGBM en el 1.3%
2. modelo 06 + LightGBM extra_trees                          0.9454  (-0.0003, mejora en 0/5)
2. LightGBM extra_trees con SVD, solo (vs. modelo 06)        0.9450  (-0.0007, mejora en 1/5)
   no coincide con LightGBM en el 1.7%
2. modelo 06 + LightGBM extra_trees con SVD                  0.9458  (+0.0001, mejora en 3/5)


2. LightGBM DART con SVD, solo (vs. modelo 06)               0.9445  (-0.0012, mejora en 0/5)
   no coincide con LightGBM en el 1.5%
2. modelo 06 + LightGBM DART con SVD                         0.9459  (+0.0002, mejora en 4/5)
2. CatBoost con SVD, solo (vs. modelo 06)                    0.9443  (-0.0014, mejora en 1/5)
   no coincide con LightGBM en el 1.9%
2. modelo 06 + CatBoost con SVD                              0.9460  (+0.0003, mejora en 4/5)
2. modelo 06 + CatBoost con SVD + DART                       0.9460  (+0.0003, mejora en 4/5)


**Lectura:** las variantes que usan la SVD son casi tan buenas como LightGBM y se equivocan en lugares algo distintos, así que como tercer o cuarto modelo **suman un poquito**. Pero la mejora queda por debajo del ruido y no se da en todas las particiones. No justifica agregar un modelo más (y CatBoost es el más lento), así que **el modelo final sigue siendo el del 06**.

## 3. Resumen

In [6]:
pd.DataFrame(resultados)

,experimento,accuracy,vs. referencia,mejora en,diferencia por partición
0,1. modelo 06 con prob_emb y vecinos en los dos...,0.9453,-0.0004,1/5,-0.0006 -0.0008 -0.0002 -0.0009 +0.0003
1,1. modelo 06 + tercer LightGBM con el embeddin...,0.9454,-0.0003,1/5,-0.0006 -0.0001 -0.0008 -0.0002 +0.0002
2,"2. LightGBM extra_trees, solo (vs. modelo 06)",0.9436,-0.0021,0/5,-0.0024 -0.0034 -0.0024 -0.0008 -0.0014
3,2. modelo 06 + LightGBM extra_trees,0.9454,-0.0003,0/5,-0.0003 -0.0004 -0.0004 -0.0002 -0.0004
4,"2. LightGBM extra_trees con SVD, solo (vs. mod...",0.9450,-0.0007,1/5,-0.0006 -0.0010 -0.0010 +0.0001 -0.0007
5,2. modelo 06 + LightGBM extra_trees con SVD,0.9458,0.0001,3/5,+0.0006 -0.0004 -0.0001 +0.0001 +0.0004
6,"2. LightGBM DART con SVD, solo (vs. modelo 06)",0.9445,-0.0012,0/5,-0.0018 -0.0007 -0.0006 -0.0014 -0.0014
7,2. modelo 06 + LightGBM DART con SVD,0.9459,0.0002,4/5,-0.0006 +0.0002 +0.0004 +0.0001 +0.0006
8,"2. CatBoost con SVD, solo (vs. modelo 06)",0.9443,-0.0014,1/5,-0.0015 -0.0019 +0.0009 -0.0023 -0.0022
9,2. modelo 06 + CatBoost con SVD,0.9460,0.0003,4/5,+0.0002 +0.0006 +0.0010 -0.0005 +0.0001


Con estas dos ideas, la validación queda planchada alrededor de **0,9457–0,9460**. Ni un modelo de lenguaje ni otros algoritmos mejoran al promedio de los dos LightGBM del 06 más allá del ruido. **No se genera una submission nueva.**